# Inspect Current ND Experiment — live probe notebook

Interactive probe of the **currently configured ND experiment definition** via raw
NIS macro calls — one small macro per question, each an independent `nis_ar -mw`
run, so a failure in one query doesn't affect the others (a crashing getter
can't take down the rest).

**Settled (live 20261027):** the time-phase getters (`ND_GetTimeLapsePhaseCount`,
`ND_GetTimePhaseSchedule`) read the **ND Acquisition Time-tab phase table** —
a different table from the sequential stimulation phases. Ground truth: after
appending 3 known stimulation phases, the getters still reported the timelapse
tab's single phase. The manual's stimulation API is setters/runners only; the
`#STRINGS` binary inventory confirms no stimulation-phase getter exists at all
in this NIS build. Consequence (TODO, not yet implemented): the pipeline must
track stimulation-phase timing itself in Python from its own phase
configuration (`ND_StimulationAppendPhase` calls).

Run **at the microscope workstation** with NIS-Elements open. The read-only
sections come first; the last section is **destructive** (it resets the
configured stimulation phases) — see its warning.

Recorded results belong in `docs/SESSION_HISTORY.md`; confirmed behavior in
`docs/NIS_REFERENCE.md`.


In [1]:
from autofrap.microscope import nis as nis_util

NIS_EXE = r'C:\Program Files\NIS-Elements\nis_ar.exe'
IN = nis_util.INI_PLACEHOLDER


def ask(label, body, keys, section='q'):
    """Run one small macro; return dict of the requested ini keys ({} on failure)."""
    try:
        cfg = nis_util._run_macro(NIS_EXE, body, ini=True)
        return {k: cfg[section][k] for k in keys}
    except Exception as e:
        print('  %s: FAILED (%s: %s)' % (label, type(e).__name__, e))
        return {}


## Read-only: ND Acquisition tab states

`ND_IsAcqTabChecked("<tab>")` — verified tab names (case sensitive; the
multichannel tab displays as λ): Time, XY, Z, Lambda, Large Image.


In [2]:
TABS = ['Time', 'XY', 'Z', 'Lambda', 'Large Image']

for tab in TABS:
    res = ask('tab %r' % tab,
              'Int_SetKeyValue("%s","q","on",ND_IsAcqTabChecked("%s"));' % (IN, tab),
              ['on'])
    if res:
        print('  %-12s active=%s' % (tab, res['on']))


  Time         active=0
  XY           active=0
  Z            active=0
  Lambda       active=1
  Large Image  active=0


## Read-only: time phases — the core question

`ND_GetTimeLapsePhaseCount()` — number of time phases in the current ND
experiment. `ND_GetTimePhaseSchedule(i, &interval, &duration, &loopcnt)` —
per-phase interval [ms], duration [ms], image count. Note: the getter does
**not** return the phase type (acquisition / stimulation / bleaching /
waiting), only timing + loop count.

Not usable here: `ND_GetExperimentLoopSize` queries the *open document* only
(-9 with no document open).


In [3]:
res = ask('phase count',
          'Int_SetKeyValue("%s","q","phases",ND_GetTimeLapsePhaseCount());' % IN,
          ['phases'])
N_PHASES = int(res['phases']) if res else 0
print('time phases in current ND experiment:', N_PHASES)


time phases in current ND experiment: 1


In [4]:
for i in range(N_PHASES):
    body = '''
double interval, duration;
int loopcnt;
ND_GetTimePhaseSchedule(%d, &interval, &duration, &loopcnt);
Int_SetKeyValue("%s","q","interval",interval);
Int_SetKeyValue("%s","q","duration",duration);
Int_SetKeyValue("%s","q","loopcnt",loopcnt);
''' % (i, IN, IN, IN)
    res = ask('phase %d schedule' % i, body, ['interval', 'duration', 'loopcnt'])
    if res:
        print('  phase %d: interval=%s ms  duration=%s ms  loops=%s'
              % (i, res['interval'], res['duration'], res['loopcnt']))
if not N_PHASES:
    print('  (no phases)')


  phase 0: interval=1000 ms  duration=10000 ms  loops=11


## Read-only: multipoint + Z series

`ND_MP_GetCount()` — multipoint position count. `ND_GetZSeriesExp(...)` —
Z-series settings of the current ND Acquisition definition. Loop params
persist even when the tab is unticked — tab state and params are separate
questions.


In [5]:
res = ask('position count',
          'Int_SetKeyValue("%s","q","count",ND_MP_GetCount());' % IN, ['count'])
if res:
    print('multipoint positions:', res['count'])

body = '''
int ztype, zcount, zhome_def, zclose;
double ztop, zhome, zbottom, zstep;
char zdevice[256];
char before[256];
char after[256];
ND_GetZSeriesExp(&ztype, &ztop, &zhome, &zbottom, &zstep, &zcount, &zhome_def, &zclose, &zdevice, &before, &after);
Int_SetKeyValue("%s","q","ztype",ztype);
Int_SetKeyValue("%s","q","ztop",ztop);
Int_SetKeyValue("%s","q","zbottom",zbottom);
Int_SetKeyValue("%s","q","zstep",zstep);
Int_SetKeyValue("%s","q","zcount",zcount);
Int_SetKeyString("%s","q","zdevice",zdevice);
''' % (IN, IN, IN, IN, IN, IN)
res = ask('z series', body,
          ['ztype', 'ztop', 'zbottom', 'zstep', 'zcount', 'zdevice'])
if res:
    print('  type=%s top=%s bottom=%s step=%s count=%s device=%r'
          % (res['ztype'], res['ztop'], res['zbottom'], res['zstep'],
             res['zcount'], res['zdevice']))


multipoint positions: 4
  type=2 top=15 bottom=-15 step=0.3 count=101 device='NIDAQ Piezo Z (name: Piezo Z)'


## Channels — DO NOT RUN (getter is a NIS-crasher)

**`ND_GetLambdaChannel(i, ...)` crashes NIS Elements on every call — do not
use it.** Confirmed live 20260904 + 20261027:

- the call signature is verbatim correct per the manual (9 params), so this is
  a bug in this NIS build's getter implementation (likely memory corruption
  while filling the out-params);
- **every call kills the NIS process, including "successful" ones**: a
  channel-0 result (only if the channel is ticked active) arrives intact
  because the ini writes complete before the process dies — the next macro
  call then finds no live GUI and spawns a fresh NIS instance, which times out
  on startup;
- for index >= 1 the crash happens inside the getter, before any writes;
- applying ND state makes no difference (channels ticked + OK; also tested
  right after running a 2-channel acquisition).

The probe body below is left commented out purely as a record of what was
tried; its last output is kept. Channel ground truth: our own
`ND_AppendLambdaChannel` configuration.


In [7]:
# DO NOT RUN — crashes NIS on every call, even when a result arrives (20260904
# + 20261027, see the note above). Kept commented out as a record of the probe.
#for i in range(8):
#    body = '''
#char name[256];
#char oc[256];
#char before[256];
#char after[256];
#int color, aftype, afarg1, afarg2;
#strcpy(&name, "SENTINEL");
#ND_GetLambdaChannel(%d, &name, &oc, &color, &before, &after, &aftype, &afarg1, &afarg2);
#Int_SetKeyString("%s","q","name",name);
#Int_SetKeyString("%s","q","oc",oc);
#''' % (i, IN, IN)
#    res = ask('channel %d' % i, body, ['name', 'oc'])
#    if not res or res['name'] in ('SENTINEL', ''):
#        break
#    print('  channel %d: name=%r oc=%r' % (i, res['name'], res['oc']))


  channel 0: name='488 CSU-W1 FRAP' oc='488 CSU-W1 FRAP'
  channel 1: FAILED (KeyError: 'q')


## Stimulation ground truth — DESTRUCTIVE

The cells below call `ND_StimulationResetPhases()`, which **removes all phases
of the current sequential stimulation experiment** — i.e. the GUI template the
pipeline runs (`run_stimulation_experiment` runs "the currently configured"
experiment). Afterwards the template is gone and must be reconfigured in the
GUI. (Per user: fine for one-off exploration.)

**Result (live 20261027):** Variant B was run — after `ND_StimulationResetPhases`
+ appending 3 known phases, the getters still reported 1 phase matching the
ND Acquisition Time tab, not the stimulation phases. The setters themselves
worked (phases appended fine). Conclusion: the time-phase getters read a
different table; there is **no macro getter for stimulation phases**. The cells
below are kept as the record of the ground-truth test.


In [8]:
# Known ground truth appended to the sequential stimulation experiment:
PHASES = [
    (0, 1000.0, 5000.0),   # 0 acquisition, interval 1000 ms, duration 5000 ms
    (1,    0.0,  200.0),   # 1 stimulation, no delay,       duration  200 ms
    (2,    0.0,  500.0),   # 2 bleaching,   no delay,       duration  500 ms
]

body = 'ND_StimulationResetPhases();\n'
for ptype, interval, duration in PHASES:
    body += 'ND_StimulationAppendPhase(%d, %s, %s);\n' % (ptype, interval, duration)
body += 'Int_SetKeyValue("%s","q","done",1);' % IN
res = ask('reset + append known phases', body, ['done'])
print('phases appended:', res)


phases appended: {'done': '1'}


In [9]:
res = ask('phase count (after append)',
          'Int_SetKeyValue("%s","q","phases",ND_GetTimeLapsePhaseCount());' % IN,
          ['phases'])
n = int(res['phases']) if res else 0
print('getters report %d time phases (expected %d)' % (n, len(PHASES)))

for i in range(n):
    body = '''
double interval, duration;
int loopcnt;
ND_GetTimePhaseSchedule(%d, &interval, &duration, &loopcnt);
Int_SetKeyValue("%s","q","interval",interval);
Int_SetKeyValue("%s","q","duration",duration);
Int_SetKeyValue("%s","q","loopcnt",loopcnt);
''' % (i, IN, IN, IN)
    r = ask('phase %d schedule' % i, body, ['interval', 'duration', 'loopcnt'])
    print('phase %d:' % i)
    if i < len(PHASES):
        ptype, interval, duration = PHASES[i]
        print('  expected : type=%d interval=%s ms duration=%s ms'
              % (ptype, interval, duration))
    else:
        print('  expected : (out of range)')
    if r:
        print('  getter   : interval=%s ms duration=%s ms loops=%s'
              % (r['interval'], r['duration'], r['loopcnt']))
    else:
        print('  getter   : FAILED')


getters report 1 time phases (expected 3)
phase 0:
  expected : type=0 interval=1000.0 ms duration=5000.0 ms
  getter   : interval=1000 ms duration=10000 ms loops=11
